# Проверка censor pipeline на реальных логах

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())
print("reports exists:", (PROJECT_ROOT / "reports").exists())


PROJECT_ROOT: /Users/anzelikasahautdinova/censor_nlp_project
src exists: True
reports exists: True


In [46]:
LOGS_PATH = PROJECT_ROOT / "data" / "logs_to_test.csv"


In [47]:
df = pd.read_csv(LOGS_PATH, sep=";", engine="python", on_bad_lines="skip")

In [48]:
df.rename(columns={'Toxic': 'toxic'}, inplace=True)


## Предобработка текста



In [52]:
RANDOM_STATE = 42
LABEL_COL = "toxic"
TEXT_COL = "Comment"

toxic_df = df[df[LABEL_COL] == 1]
safe_df = df[df[LABEL_COL] == 0]

safe_sample = safe_df.sample(
    n=len(toxic_df),
    random_state=RANDOM_STATE
)

df = pd.concat(
    [toxic_df, safe_sample],
    axis=0
).sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

print(df[LABEL_COL].value_counts())
print(df.shape)

toxic
1    308
0    308
Name: count, dtype: int64
(616, 2)


In [53]:
from src.preprocess import clean_text

df["clean_text"] = df[TEXT_COL].apply(clean_text)
df[[TEXT_COL, "clean_text"]].head()


,Comment,clean_text
0,Уроды.,уроды.
1,Самый тупорылый банк!!!!,самый тупорылый банк!!!!
2,Как пополнить счет в обход ареста,как пополнить счет в обход ареста
3,А что такое??? Почему твою мать мне не приходит код для оплаты???? Твари!!!! Где кода????? Не приходит ничего достали уже гады! Где кооооооооод????,а что такое??? почему твою мать мне не приходит код для оплаты???? твари!!!! где кода????? не приходит ничего достали уже гады! где кооооооооод????
4,Если ваша тварь мне будет звонить пойдет на х,если ваша тварь мне будет звонить пойдет на х


## Запуск Regex only


In [54]:
from src.regex_filter import RegexToxicFilter

regex_model = RegexToxicFilter()

regex_outputs = regex_model.predict_batch(df["clean_text"].tolist())

df["regex_pred"] = [x["is_toxic"] for x in regex_outputs]
df["regex_score"] = [x["score"] for x in regex_outputs]
df["regex_matches"] = [", ".join(x["matches"]) for x in regex_outputs]

df[[TEXT_COL, "regex_pred", "regex_matches"]].head(20)


,Comment,regex_pred,regex_matches
0,Уроды.,0,
1,Самый тупорылый банк!!!!,0,
2,Как пополнить счет в обход ареста,0,
3,А что такое??? Почему твою мать мне не приходит код для оплаты???? Твари!!!! Где кода????? Не приходит ничего достали уже гады! Где кооооооооод????,0,
4,Если ваша тварь мне будет звонить пойдет на х,0,
5,"Адрес этих дубиноголовых: Калининград, ул. Горького 000.",0,
6,Ты в натуре долбаеб,0,
7,У нас в городе проблема с вашими банкоматами. Где ещё можно снять наличные с кредитной карты без комиссии?,0,
8,Ваши операторы говно кроме Натальи проследите по звонкам,0,
9,"Ни в одном банке кредитку не заблокировали ни сбер ни Т банк, только в вашем ублюдском банке , в котором еще и переводом нельзя именно кредитку оплатить",0,


## Запуск RuBERT


In [58]:
from tqdm.auto import tqdm
from src.bert_filter import BertToxicFilter

bert_model = BertToxicFilter()

bert_outputs = []
for text in tqdm(df["clean_text"].tolist()):
    bert_outputs.append(bert_model.predict_one(text))

df["bert_pred"] = [x["is_toxic"] for x in bert_outputs]
df["bert_score"] = [x["score"] for x in bert_outputs]
df["bert_label"] = [x["label"] for x in bert_outputs]

df[[TEXT_COL, "bert_pred", "bert_label", "bert_score"]].head(20)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: sismetanin/rubert-toxic-pikabu-2ch
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/616 [00:00<?, ?it/s]

,Comment,bert_pred,bert_label,bert_score
0,Уроды.,1,LABEL_1,0.991298
1,Самый тупорылый банк!!!!,1,LABEL_1,0.994160
2,Как пополнить счет в обход ареста,0,LABEL_0,0.996592
3,А что такое??? Почему твою мать мне не приходит код для оплаты???? Твари!!!! Где кода????? Не приходит ничего достали уже гады! Где кооооооооод????,1,LABEL_1,0.878746
4,Если ваша тварь мне будет звонить пойдет на х,1,LABEL_1,0.991832
5,"Адрес этих дубиноголовых: Калининград, ул. Горького 000.",1,LABEL_1,0.980355
6,Ты в натуре долбаеб,1,LABEL_1,0.917995
7,У нас в городе проблема с вашими банкоматами. Где ещё можно снять наличные с кредитной карты без комиссии?,0,LABEL_0,0.997574
8,Ваши операторы говно кроме Натальи проследите по звонкам,0,LABEL_0,0.983151
9,"Ни в одном банке кредитку не заблокировали ни сбер ни Т банк, только в вашем ублюдском банке , в котором еще и переводом нельзя именно кредитку оплатить",0,LABEL_0,0.994767


## Запуск Hybrid


In [59]:
from src.hybrid_filter import HybridToxicFilter

hybrid_model = HybridToxicFilter()

hybrid_outputs = []
for text in tqdm(df["clean_text"].tolist()):
    hybrid_outputs.append(hybrid_model.predict_one(text))

df["hybrid_pred"] = [x["is_toxic"] for x in hybrid_outputs]
df["hybrid_score"] = [x["score"] for x in hybrid_outputs]
df["hybrid_matches"] = [", ".join(x.get("regex_matches", [])) for x in hybrid_outputs]

df[[TEXT_COL, "regex_pred", "bert_pred", "hybrid_pred", "regex_matches"]].head(20)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: sismetanin/rubert-toxic-pikabu-2ch
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/616 [00:00<?, ?it/s]

,Comment,regex_pred,bert_pred,hybrid_pred,regex_matches
0,Уроды.,0,1,1,
1,Самый тупорылый банк!!!!,0,1,1,
2,Как пополнить счет в обход ареста,0,0,0,
3,А что такое??? Почему твою мать мне не приходит код для оплаты???? Твари!!!! Где кода????? Не приходит ничего достали уже гады! Где кооооооооод????,0,1,1,
4,Если ваша тварь мне будет звонить пойдет на х,0,1,1,
5,"Адрес этих дубиноголовых: Калининград, ул. Горького 000.",0,1,1,
6,Ты в натуре долбаеб,0,1,1,
7,У нас в городе проблема с вашими банкоматами. Где ещё можно снять наличные с кредитной карты без комиссии?,0,0,0,
8,Ваши операторы говно кроме Натальи проследите по звонкам,0,0,0,
9,"Ни в одном банке кредитку не заблокировали ни сбер ни Т банк, только в вашем ублюдском банке , в котором еще и переводом нельзя именно кредитку оплатить",0,0,0,


## Сравнение моделей на логах


In [60]:
summary = pd.DataFrame({
    "model": ["Regex", "RuBERT", "Hybrid"],
    "predicted_toxic_count": [
        int(df["regex_pred"].sum()),
        int(df["bert_pred"].sum()),
        int(df["hybrid_pred"].sum()),
    ],
    "predicted_toxic_share": [
        df["regex_pred"].mean(),
        df["bert_pred"].mean(),
        df["hybrid_pred"].mean(),
    ]
})

summary


,model,predicted_toxic_count,predicted_toxic_share
0,Regex,85,0.137987
1,RuBERT,313,0.508117
2,Hybrid,313,0.508117


In [62]:
df

,Comment,toxic,clean_text,regex_pred,regex_score,regex_matches,hybrid_pred,hybrid_score,hybrid_matches,bert_pred,bert_score,bert_label
0,Уроды.,1,уроды.,0,0.0,,1,0.991298,,1,0.991298,LABEL_1
1,Самый тупорылый банк!!!!,1,самый тупорылый банк!!!!,0,0.0,,1,0.994160,,1,0.994160,LABEL_1
2,Как пополнить счет в обход ареста,0,как пополнить счет в обход ареста,0,0.0,,0,0.003408,,0,0.996592,LABEL_0
3,А что такое??? Почему твою мать мне не приходит код для оплаты???? Твари!!!! Где кода????? Не приходит ничего достали уже гады! Где кооооооооод????,1,а что такое??? почему твою мать мне не приходит код для оплаты???? твари!!!! где кода????? не приходит ничего достали уже гады! где кооооооооод????,0,0.0,,1,0.878746,,1,0.878746,LABEL_1
4,Если ваша тварь мне будет звонить пойдет на х,1,если ваша тварь мне будет звонить пойдет на х,0,0.0,,1,0.991832,,1,0.991832,LABEL_1
...,...,...,...,...,...,...,...,...,...,...,...,...
611,Ты долбаеб,1,ты долбаеб,0,0.0,,1,0.959336,,1,0.959336,LABEL_1
612,Пиздец ты ебан,1,пиздец ты ебан,1,1.0,"пиздец, ебан, пиздец",1,1.000000,"пиздец, ебан, пиздец",1,0.966517,LABEL_1
613,"Вчера вы писали, что вопрос решится в течении суток!!! Верните деньги, мошенники!!!",1,"вчера вы писали, что вопрос решится в течении суток!!! верните деньги, мошенники!!!",0,0.0,,0,0.071744,,0,0.928256,LABEL_0
614,Не тупой бот,0,не тупой бот,0,0.0,,1,0.982859,,1,0.982859,LABEL_1


In [66]:
error_examples = df[
    (
        (df["regex_pred"] != df["toxic"]) |
        (df["bert_pred"] != df["toxic"])
    )
][[
    "clean_text",
    "toxic",
    "regex_pred",
    "bert_pred",
    "hybrid_pred"
]].head(70)

error_examples

,clean_text,toxic,regex_pred,bert_pred,hybrid_pred
0,уроды.,1,0,1,1
1,самый тупорылый банк!!!!,1,0,1,1
3,а что такое??? почему твою мать мне не приходит код для оплаты???? твари!!!! где кода????? не приходит ничего достали уже гады! где кооооооооод????,1,0,1,1
4,если ваша тварь мне будет звонить пойдет на х,1,0,1,1
5,"адрес этих дубиноголовых: калининград, ул. горького 000.",1,0,1,1
...,...,...,...,...,...
138,вы цыгане !,1,0,1,1
140,где мои деньги машенники,1,0,0,0
141,вот это что? где мои деньги?!?!? на этом счету нету денег!!!!!!! воры,1,0,0,0
143,вы дебилы. я ее закрыл в январе. прочитайте выше переписку,1,0,1,1


In [67]:
LABEL_COL = "toxic"

if LABEL_COL in df.columns:
    from src.evaluate import evaluate_predictions
    
    y_true = df[LABEL_COL].astype(int)
    
    metrics = []
    metrics.append(evaluate_predictions(y_true, df["regex_pred"], "Regex logs"))
    metrics.append(evaluate_predictions(y_true, df["bert_pred"], "RuBERT logs"))
    metrics.append(evaluate_predictions(y_true, df["hybrid_pred"], "Hybrid logs"))
    
    metrics_df = pd.DataFrame(metrics)
    display(metrics_df)




===== Regex logs =====
{'model': 'Regex logs', 'accuracy': 0.6217532467532467, 'precision': 0.9411764705882353, 'recall': 0.2597402597402597, 'f1': 0.4071246819338422}

Classification report:
              precision    recall  f1-score   support

           0       0.57      0.98      0.72       308
           1       0.94      0.26      0.41       308

    accuracy                           0.62       616
   macro avg       0.76      0.62      0.56       616
weighted avg       0.76      0.62      0.56       616


Confusion matrix:
[[303   5]
 [228  80]]

===== RuBERT logs =====
{'model': 'RuBERT logs', 'accuracy': 0.797077922077922, 'precision': 0.792332268370607, 'recall': 0.8051948051948052, 'f1': 0.7987117552334944}

Classification report:
              precision    recall  f1-score   support

           0       0.80      0.79      0.80       308
           1       0.79      0.81      0.80       308

    accuracy                           0.80       616
   macro avg       0.80    

,model,accuracy,precision,recall,f1
0,Regex logs,0.621753,0.941176,0.259740,0.407125
1,RuBERT logs,0.797078,0.792332,0.805195,0.798712
2,Hybrid logs,0.797078,0.792332,0.805195,0.798712


In [26]:
df["bert_score"].describe()

count    3001.000000
mean        0.950721
std         0.097334
min         0.503790
25%         0.966137
50%         0.993329
75%         0.996897
max         0.998248
Name: bert_score, dtype: float64

## Анализ расхождений Regex и BERT


In [35]:
regex_only_toxic = df[
    (df["regex_pred"] == 1) &
    (df["bert_pred"] == 0)
].copy()

bert_only_toxic = df[
    (df["regex_pred"] == 0) &
    (df["bert_pred"] == 1)
].copy()

both_toxic = df[
    (df["regex_pred"] == 1) &
    (df["bert_pred"] == 1)
].copy()

print("Regex toxic, BERT safe:", len(regex_only_toxic))
print("Regex safe, BERT toxic:", len(bert_only_toxic))
print("Both toxic:", len(both_toxic))


Regex toxic, BERT safe: 8
Regex safe, BERT toxic: 236
Both toxic: 77


In [45]:
regex_only_toxic[[TEXT_COL, "regex_matches", "bert_label", "bert_score"]].head(100)


,comment,regex_matches,bert_label,bert_score
118,"Вчера было 170 рублей ,проценты за кредитку,и так же вчера положил на счёт кредитки энную сумму денег,а сегодня проценты выросли,??? Это что за такое? Они должны наоборот уменьшиться,что за сука пидарский банк ваш ?",сука,LABEL_0,0.997788
162,Как связаться с человеком блять,блять,LABEL_0,0.963245
176,"Я не могу оплатить кредиты я из Курска. Думаю вы это должны знать гандоны. Я не разу не на минуту не просрочил платежи, в этот раз я вынужден",гандоны,LABEL_0,0.997994
198,"Ужас! Банк мошенников и крахаборов! Что мне надо сделать, чтобы закрыть эту ебучую карту?",ебучую,LABEL_0,0.856223
271,"Куръер конечно, полная мразь, он мне сразу не понравился, алкаш, да и банк, конечно говно полное, наебшики ебаные, да, Марин, конечно, отключаем эту хуету, я много раз операторов спрашивал, если ли платежи, не одна шкура мне не подсказала..","ебаные, мразь",LABEL_0,0.997479
305,Оформлю банкротство и пошли на хуй тогда. Раз Я должен пробиваться к оператору,"хуй, хуй",LABEL_0,0.997554
486,Уважение... Да пипец одна на ебало во. Вчера была возможность в приложении запросить рефинансирование без посещения банка а сегодня нет. Человек который должен был настроить приложение для айфона не приехал во время папе придется еще один день в этом сарае в рязани.,ебало,LABEL_0,0.998179
566,Я у вас спросил что ? Чтоб вы мне рассказали если я сниму наличные и что будет ? Мне какая разница при снятии наличных какой процент изымут там нечего нового все так же а вы мне объясните какой процент потом будет ? Или вы настолько глупая или я хуй знает где вас таки набирают походу по объявлению,"хуй, хуй",LABEL_0,0.862802


In [38]:
pd.set_option('display.max_colwidth', None)  # Or pd.options.display.max_colwidth = None


In [43]:
bert_only_toxic[[TEXT_COL, "bert_label", "bert_score"]].head(50)


,comment,bert_label,bert_score
0,"Я ничего не должен, уйобки. \nРасторгните договор, твари.",LABEL_1,0.987963
1,Опять суки деньги прокручиваете,LABEL_1,0.977816
5,Вы что идиоты?),LABEL_1,0.986459
8,Когда нибудь ваш гребанный баек будет еормально работать? Вымогатели,LABEL_1,0.996025
9,Вы наебалово. Игры ваши гейские,LABEL_1,0.993842
14,"Уроды вы, списываете хорошо, при этом нигде нет информации про такие моменты!!!",LABEL_1,0.865267
15,Дрявый банк,LABEL_1,0.854482
16,Альфа-Помощник ты бесполезен!,LABEL_1,0.908899
18,Мошенники представляются вами,LABEL_1,0.548768
19,боже альфа банк самый дерьмовый банк на пару со сбером,LABEL_1,0.856843


## 10. Сохранение результатов


In [ ]:
reports_dir = PROJECT_ROOT / "reports"
reports_dir.mkdir(exist_ok=True)

logs_df.to_csv(reports_dir / "logs_pipeline_predictions.csv", index=False)
regex_only_toxic.to_csv(reports_dir / "logs_regex_only_toxic.csv", index=False)
bert_only_toxic.to_csv(reports_dir / "logs_bert_only_toxic.csv", index=False)
both_toxic.to_csv(reports_dir / "logs_both_toxic.csv", index=False)

print("Saved:")
print(reports_dir / "logs_pipeline_predictions.csv")
print(reports_dir / "logs_regex_only_toxic.csv")
print(reports_dir / "logs_bert_only_toxic.csv")
print(reports_dir / "logs_both_toxic.csv")
